# Flight Advection

This notebook demonstrates the core functionality of the `advection` library, which is used to model the movement (advection) of aircraft waypoints over time using weather data.

In [65]:
%load_ext autoreload
%autoreload 2

import datetime
from dateutil import tz
import sys
sys.path.append('../src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pycontrails import Flight
import xarray as xr
import plotly.graph_objects as go

import advection.vectorized_prediction.advection as advection
import advection.utils.casting as casting
import advection.vectorized_prediction.time_util as time_util

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Define a Sample Flight

We define a synthetic flight segment representing an aircraft at altitude 10,668 m or 35,000 ft (~238 hPa) over the central US.

In [66]:
# Create a synthetic flight with multiple waypoints
df = pd.DataFrame({
    'time': [
        datetime.datetime(2023, 1, 1, 12, 0, 0, tzinfo=tz.UTC),
        datetime.datetime(2023, 1, 1, 12, 15, 0, tzinfo=tz.UTC),
        datetime.datetime(2023, 1, 1, 12, 30, 0, tzinfo=tz.UTC),
        datetime.datetime(2023, 1, 1, 12, 45, 0, tzinfo=tz.UTC),
        datetime.datetime(2023, 1, 1, 13, 0, 0, tzinfo=tz.UTC),
    ],
    'latitude': [40.0, 40.5, 41.0, 41.5, 42.0],
    'longitude': [-100.0, -99.0, -98.0, -97.0, -96.0],
    'altitude_ft': [35000.0, 35000.0, 36000.0, 36000.0, 37000.0],
    'flight_id': 'demo_flight'
})
fl = Flight(df)

# Convert pycontrails Flight to AdvectionState
# Here we explicitly convert the flight time to hours since epoch
time_hours = casting.timetype_to_float(fl.data['time'], unit=casting.Units.HOURS)
initial_state = advection.AdvectionState.from_flight(fl)

print(f"Created Flight with {len(fl)} waypoints.")
print(f"Initial State shape: {initial_state.pressure.shape}")


Created Flight with 5 waypoints.
Initial State shape: (5,)


## Setup Mock Weather Data

For this demo, we create a small synthetic `xarray` dataset containing wind velocities `u` (Eastward), `v` (Northward), and `w` (Vertical), and temperature `t` (K). In production, this would be loaded from regular netCDF or zarr files.

In [67]:
# Create a grid covering our flight path
lats = np.linspace(38, 44, 13)
lons = np.linspace(-102, -94, 17)
levels = np.array([200.0, 250.0, 300.0])
times = np.arange(time_hours.min() - 1, time_hours.max() + 3, 0.5)

# Create fluctuating wind blowing North-East
u_wind = (
    100.0
    + 0.5 * (levels - 250).reshape(1, -1, 1, 1)  # Vertical shear
    + 5.0 * np.sin(np.deg2rad(lats * 30)).reshape(1, 1, -1, 1) # Latitudinal variation
    + np.random.normal(scale=2.0, size=(len(times), len(levels), len(lats), len(lons))) # Noise
)
v_wind = (
    50.0
    - 0.2 * (lons + 98).reshape(1, 1, 1, -1)     # Longitudinal gradient
    + np.random.normal(scale=1.0, size=(len(times), len(levels), len(lats), len(lons))) # Noise
)
w_wind = np.random.normal(scale=0.01, size=(len(times), len(levels), len(lats), len(lons)))

# Standard temperature at flight levels (around 220 K)
t_temp = (
    220.0
    + 0.1 * (levels - 250).reshape(1, -1, 1, 1)  # Vertical lapse rate
    - 0.5 * (lats - 41).reshape(1, 1, -1, 1)    # Latitude gradient
    + 2.0 * np.sin(np.deg2rad(lons * 20)).reshape(1, 1, 1, -1) # Longitudinal wave
    + np.random.normal(scale=0.1, size=(len(times), len(levels), len(lats), len(lons))) # Noise
)

weather_ds = xr.Dataset(
    {
        "u": (["time", "level", "latitude", "longitude"], u_wind),
        "v": (["time", "level", "latitude", "longitude"], v_wind),
        "w": (["time", "level", "latitude", "longitude"], w_wind),
        "t": (["time", "level", "latitude", "longitude"], t_temp),
    },
    coords={
        "time": times,
        "level": levels,
        "latitude": lats,
        "longitude": lons,
    },
)
print("Synthetic weather bounds updated to cover flight path.")
print(
    "Latitudes:",
    weather_ds.latitude.values.min(),
    "to",
    weather_ds.latitude.values.max(),
)
print(
    "Longitudes:",
    weather_ds.longitude.values.min(),
    "to",
    weather_ds.longitude.values.max(),
)
print("Times:", weather_ds.time.values.min(), "to", weather_ds.time.values.max())


Synthetic weather bounds updated to cover flight path.
Latitudes: 38.0 to 44.0
Longitudes: -102.0 to -94.0
Times: 464603.0 to 464607.5


## Advection: Uniform Advection

We initialize the `LagrangianAdvector` with our weather data and advect each flight waypoint uniformly forward in time for 2 hours with 30-minute increments, creating a total of 5 steps (the original flight + 4 advection steps).

In [68]:
# Define evaluation times: each waypoint advected for 2 hours in 30-min steps
advection_length_hours = 2.0
step_size_secs = 1800.0

t_eval = time_util.t_eval_homogeneous(
    t_start=initial_state.time,
    advection_length_hours=advection_length_hours,
    step_size_secs=step_size_secs,
)

advector = advection.LagrangianAdvector(weather_ds)
# Advect with moist advection.
advected_steps = advector.advect(fl, t_eval, True)

for i, flight in enumerate(advected_steps):
    print(f'Displaying flight at advection step {i}.')
    display(flight)
    print('\n\n')

Displaying flight at advection step 0.


Flight [9 keys x 5 length, 0 attributes]
	Keys: time, advected_time, latitude, longitude, level, ..., flight_id
	Attributes:
	time                [2023-01-01 12:00:00, 2023-01-01 13:00:00]
	longitude           [-100.0, -96.0]
	latitude            [40.0, 42.0]
	altitude            [10668.0, 11277.6]




Displaying flight at advection step 1.


Flight [9 keys x 5 length, 0 attributes]
	Keys: time, advected_time, latitude, longitude, level, ..., flight_id
	Attributes:
	time                [2023-01-01 12:00:00, 2023-01-01 13:00:00]
	longitude           [-97.911, -94.195]
	latitude            [40.81, 42.805]
	altitude            [10668.3, 11278.2]




Displaying flight at advection step 2.


Flight [9 keys x 5 length, 0 attributes]
	Keys: time, advected_time, latitude, longitude, level, ..., flight_id
	Attributes:
	time                [2023-01-01 12:00:00, 2023-01-01 13:00:00]
	longitude           [-95.84, -92.451]
	latitude            [41.612, 43.592]
	altitude            [10667.9, 11279.5]




Displaying flight at advection step 3.


Flight [9 keys x 5 length, 0 attributes]
	Keys: time, advected_time, latitude, longitude, level, ..., flight_id
	Attributes:
	time                [2023-01-01 12:00:00, 2023-01-01 13:00:00]
	longitude           [-93.806, -90.705]
	latitude            [42.411, 44.394]
	altitude            [10669.4, 11279.2]




Displaying flight at advection step 4.


Flight [9 keys x 5 length, 0 attributes]
	Keys: time, advected_time, latitude, longitude, level, ..., flight_id
	Attributes:
	time                [2023-01-01 12:00:00, 2023-01-01 13:00:00]
	longitude           [-91.784, -88.895]
	latitude            [43.2, 45.2]
	altitude            [10668.8, 11285.7]

## Advection: Advecting a Flight to Time

We initialize the `LagrangianAdvector` with our weather data and uniformly advect all the flight waypoints forward to a specific point in time (13:30 UTC for this example), creating a total of 2 steps (the original flight + 1 advection step).

In [69]:
# Define evaluation time.
target_time = pd.Timestamp('2023-01-01 13:30:00', tz='UTC')
target_hrs = target_time.timestamp() / 3600

t_eval = np.stack([
      initial_state.time,
      np.full_like(initial_state.time, target_hrs),
  ])

advector = advection.LagrangianAdvector(weather_ds)
# Advect with moist advection.
advected_steps = advector.advect(fl, t_eval, True)

for i, flight in enumerate(advected_steps):
    print(f'Displaying flight at advection step {i}.')
    display(flight)
    print('\n\n')

Displaying flight at advection step 0.


Flight [9 keys x 5 length, 0 attributes]
	Keys: time, advected_time, latitude, longitude, level, ..., flight_id
	Attributes:
	time                [2023-01-01 12:00:00, 2023-01-01 13:00:00]
	longitude           [-100.0, -96.0]
	latitude            [40.0, 42.0]
	altitude            [10668.0, 11277.6]




Displaying flight at advection step 1.


Flight [9 keys x 5 length, 0 attributes]
	Keys: time, advected_time, latitude, longitude, level, ..., flight_id
	Attributes:
	time                [2023-01-01 12:00:00, 2023-01-01 13:00:00]
	longitude           [-94.197, -93.78]
	latitude            [42.408, 42.804]
	altitude            [10668.3, 11278.7]

## Visualize the Trajectory

Trace how the flight moved over the advection period. Note that this visualization works with either the 'Uniform Advection' or 'Advecting a Flight to a Time' cells.

In [70]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np

fig = go.Figure()

colors = px.colors.qualitative.Plotly
parcel_paths = {}

for i, flight in enumerate(advected_steps):
    df = flight.dataframe.copy()
    
    df['time'] = pd.to_datetime(df['time'])
    if 'advected_time' in df.columns:
        df['advected_time'] = pd.to_datetime(df['advected_time'])
    else:
        df['advected_time'] = df['time']
    
    if 'parcel_id' in df.columns:
        for _, row in df.iterrows():
            pid = row['parcel_id']
            if pid not in parcel_paths:
                parcel_paths[pid] = {'x': [], 'y': [], 'z': []}
            parcel_paths[pid]['x'].append(row['longitude'])
            parcel_paths[pid]['y'].append(row['latitude'])
            parcel_paths[pid]['z'].append(row['level'])
            
    time_diff_minutes = ((df['advected_time'] - df['time']).dt.total_seconds() / 60.0).round().astype(int)
    labels = [f"+{m}m" if m > 0 else "" for m in time_diff_minutes]
    name = f'Advected Step {i}' if i > 0 else 'Original (unadvected) flight'
    color = colors[i % len(colors)]

    fig.add_trace(go.Scatter3d(
        x=df['longitude'],
        y=df['latitude'],
        z=df['level'],
        mode='lines+markers+text',
        marker=dict(size=4, symbol='circle'),
        line=dict(color=color, width=3),
        text=labels,
        textposition="top center",
        textfont=dict(
            family="sans-serif",
            size=10,
            color=color
        ),
        customdata=df['parcel_id'] if 'parcel_id' in df.columns else None,
        name=name
    ))

# Draw travel lines between points as they get advected through time
for pid, coords in parcel_paths.items():
    if len(coords['x']) > 1:
        fig.add_trace(go.Scatter3d(
            x=coords['x'],
            y=coords['y'],
            z=coords['z'],
            mode='lines',
            line=dict(color='gray', width=1.5, dash='dash'),
            hoverinfo='none',
            showlegend=False
        ))

fig.update_layout(
    width=1100,
    height=800,
    scene=dict(
        xaxis_title='Longitude',
        yaxis_title='Latitude',
        zaxis_title='Level (hPa)',
    ),
    title={
        'text': "Multi-Waypoint Advection (Interactive 3D)",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    legend=dict(
        yanchor="top", 
        y=0.95, 
        xanchor="left", 
        x=0.01
    )
)

fig.show()
